In [1]:
import os
import re
import sys
import pickle
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from multiprocessing import Pool

tqdm.pandas()

os.environ['CUDA_VISIBLE_DEVICES'] = '1'

path = '/slow-data/journalisticfeelings/' #@param {"type": "string"}
os.chdir(path)

articles_path = './data/nexis-uni-export-unique.p3' #@param {"type": "string"}
predictions_path = './data/predictions/2023-02-13/' #@param {"type": "string"}
pre_annotations_path = './data/preannotations/2023-02-13/' #@param {"type": "string"}

In [ ]:
#@markdown # Load articles

articles = pd.read_pickle(articles_path)

In [2]:
#@markdown # Load POS, quotes and dependencies

tokens = pd.concat([
  pd.read_pickle(f'{predictions_path}/{fn}')
  for fn in tqdm(os.listdir(predictions_path))
  # if int(fn[6:-3]) < 10
]).reset_index(['paragraph', 'sentence'])

tokens['paragraph'] = (~tokens['text'].isna()).cumsum()
tokens.set_index(['paragraph', 'sentence'], inplace=True)
tokens.reset_index(inplace=True)
tokens['paragraph'] = tokens['token'].isna().cumsum()
tokens['token idx'] = tokens.groupby(['paragraph', 'sentence']).cumcount()
tokens.set_index(['paragraph', 'sentence', 'token idx'], inplace=True)

text_to_paragraph = tokens['text'].dropna().reset_index(
  'sentence', drop=True).reset_index().set_index(
  'text')['paragraph'].to_dict()

  0%|          | 0/502 [00:00<?, ?it/s]

In [ ]:
#@markdown ## Helper function

def to_annotation(article, pronoun_tokens):
  global tokens, text_to_paragraph
  article_tokens = tokens.loc[[
      text_to_paragraph[paragraph]
      for paragraph in article['body']
  ]]
  article_tokens.reset_index(inplace=True)
  article_tokens['paragraph'] = article_tokens['token'].isna().cumsum() -1
  article_tokens.set_index(['paragraph', 'sentence'], inplace=True)

  data = [
    {'text': paragraph, 'paragraph': f'{num}'}
    for num, paragraph in enumerate(article_tokens['text'].dropna())
  ]

  article_tokens = article_tokens[~article_tokens['token'].isna()].drop('text', axis=1)
  pronouns = pd.DataFrame({
      pronoun_type: article_tokens['token'].str.lower().isin(pronouns)
      for pronoun_type, pronouns in pronoun_tokens.items()
  })

  personal = pronouns[article_tokens['quote']==False].max().max()
  personal = personal if personal==personal else False

  annotations = [{
    "from_name": 'titlelabels', "to_name": 'title', "type": "choices",
    'value': {"choices": [["NOT PERSONAL", "PERSONAL"][int(personal)]]},
  }] + [
    {
    "from_name": 'quotes', "to_name": 'paragraphs', "type": "paragraphlabels",
    'value': {
        'paragraphlabels': [['PERSONAL', 'IGNORED'][int(token['quote'])]],
        'start': str(int(paragraph)), 'end': str(int(paragraph)), 'text': "", #token['token'],
        'startOffset': int(token['start']), 'endOffset': int(token['end'])
      }
    }
    for (paragraph, _), token in article_tokens[pronouns.max(1)].iterrows()
  ] + [
    {
    "from_name": 'quotes', "to_name": 'paragraphs', "type": "paragraphlabels",
    'value': {
        'paragraphlabels': [token['pronoun']],
        'start': str(int(paragraph)), 'end': str(int(paragraph)), 'text': "", #token['token'],
        'startOffset': int(token['start']), 'endOffset': int(token['end'])
      }
    }
    for (paragraph, _), token in article_tokens[~article_tokens['pronoun'].isna()].iterrows()
  ] + [
    {
    "from_name": 'quotes', "to_name": 'paragraphs', "type": "paragraphlabels",
    'value': {
        'paragraphlabels': ['QUOTE'], 'text': "", #article['body'][paragraph][first:last],
        'start': str(int(paragraph)), 'end': str(int(paragraph)),
        'startOffset': first, 'endOffset': last,
      }
    } 
    for (paragraph, _), quote_tokens in article_tokens.groupby(
      ['paragraph', 'quote idx'])
    for first, last in [[int(quote_tokens.iloc[0]['start']),
                        int(quote_tokens.iloc[-1]['end'])]]
  ]
  return {
      'predictions': [
          {"model_version": "personal_quotes_model", "result": annotations},
      ],
      'data': {
          'paragraphs': data,
          'title': ' '.join(article['title']),
          'zip': article['zip'],
          'docx': article['docx'],
      }
  }


In [ ]:
#@markdown # Export annotations

n_jobs = 10 #@param {"type": "integer"}

results_path = './results/2022-12-15/'

pronoun_tokens = 'IK:mijn,mezelf,me,mij,ik,ikzelf,mijne,WIJ:onze,onszelf,wij,ons,we' #@param {"type": "string"}
# Seperate pronouns for IK and WE using the funky syntax IK:a,b,c,...-syntax
pronoun_tokens = pd.Series(pronoun_tokens.split(',')).str[::-1].str.split(':', expand=True).ffill().applymap(lambda x: x[::-1]).groupby(1)[0].apply(list).to_dict()

batch_size = 1024 #@param {"type": "integer"}

with Pool(n_jobs) as pool:
  batches = articles.groupby((np.arange(len(articles)) // batch_size))
  for batch_num, batch in tqdm(batches, total=batches.ngroups):
    tqdm.pandas(desc=f'Batch {batch_num}', leave=False)
    promises = batch.apply(
      lambda x: pool.apply_async(to_annotation, (x.copy(), ), {'pronoun_tokens': pronoun_tokens}), axis=1)
    data = promises.progress_apply(lambda x: x.get()).tolist()
    data_ = dict()
    for sample in data:
      c = sample['predictions'][0]['result'][0]['value']['choices'][0]
      data_[c] = data_.get(c, [])
      data_[c].append(sample)
    with open(f'{results_path}/personal-labels-batch-{batch_num:04d}.json', 'w') as f:
      json.dump(data_['PERSONAL'], f)
    with open(f'{results_path}/not-personal-labels-batch-{batch_num:04d}.json', 'w') as f:
      json.dump(data_['NOT PERSONAL'], f)

  0%|          | 0/61 [00:00<?, ?it/s]

Batch 0:   0%|          | 0/1024 [00:00<?, ?it/s]

In [ ]:
#@markdown # Run one test prediction

from flair.data import Sentence
from flair.models import SequenceTagger
from flair.visual.ner_html import render_ner_html
from IPython.display import display, HTML

if 'model' not in globals():
  model = SequenceTagger.load("resources/taggers/journalistic-quote-detection-xlm-roberta-base-set-350/final-model.pt")

text = "Als wij 's middags thuiskom van school, zit mijn vader op de bank gebiologeerd naar de televisie te kijken. 'Het is lastig om vanuit de achtertuin een blik op de tv te werpen.' Ik zie een zwarte streep dwars door het midden van het scherm lopen. Daaromheen is alles wit. Over die zwarte streep lijken een paar stipjes zich moeizaam voort te bewegen. Met de fiets nog tussen de benen, ik vermoed een grote gebeurtenis." #@param {"type": "string"}

s = Sentence(text)

model.predict(s)


HTML(render_ner_html(s, label_name="quote"))